In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/credit_data_clean.csv")
print(df.shape)
df.head()

(1860331, 89)


,Unnamed: 0,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,debt_settlement_flag,target
0,0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,10+ years,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0
1,1,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,< 1 year,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,1
2,2,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,10+ years,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0
3,3,10000.0,10000.0,10000.0,36 months,13.49%,339.31,C,C1,10+ years,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0
4,4,3000.0,3000.0,3000.0,60 months,12.69%,67.79,B,B5,1 year,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0


In [4]:
df_features = df.copy()
    
print("Creating domain-specific features...")
    
    # 1. Debt-to-Income Ratio
if 'annual_inc' in df_features.columns and 'loan_amnt' in df_features.columns:
    df_features['debt_to_income_ratio'] = df_features['loan_amnt'] / (df_features['annual_inc'] + 1)
    print("  - Created debt_to_income_ratio")
    
    # 2. Credit Utilization
if 'revol_bal' in df_features.columns and 'revol_util' in df_features.columns:
        df_features['revol_util'] = pd.to_numeric(
            df_features['revol_util'].astype(str).str.replace('%', ''), errors='coerce'
        )
        df_features['credit_utilization'] = df_features['revol_util'] / 100
        print("  - Created credit_utilization")
        
    # 3. Employment Length (convert to numeric)
if 'emp_length' in df_features.columns:
        df_features['emp_length_years'] = df_features['emp_length'].astype(str).str.extract('(\d+)').astype(float)
        df_features['emp_length_years'] = df_features['emp_length_years'].fillna(0)
        print("  - Created emp_length_years")
    
    # 4. FICO Score Average
if 'fico_range_low' in df_features.columns and 'fico_range_high' in df_features.columns:
        df_features['fico_avg'] = (df_features['fico_range_low'] + df_features['fico_range_high']) / 2
        df_features['fico_range_width'] = df_features['fico_range_high'] - df_features['fico_range_low']
        print("  - Created fico_avg and fico_range_width")
    
    # 5. Loan Amount Categories
if 'loan_amnt' in df_features.columns:
        df_features['loan_amount_category'] = pd.cut(df_features['loan_amnt'], 
                                                   bins=[0, 10000, 20000, 35000, float('inf')],
                                                   labels=['Small', 'Medium', 'Large', 'Very_Large'])
        print("  - Created loan_amount_category")
    
    # 6. Interest Rate Categories
if 'int_rate' in df_features.columns:
        df_features['int_rate'] = pd.to_numeric(
            df_features['int_rate'].astype(str).str.replace('%', ''), errors='coerce'
        )

        df_features['int_rate_category'] = pd.cut(df_features['int_rate'],
                                                bins=[0, 10, 15, 20, float('inf')],
                                                labels=['Low', 'Medium', 'High', 'Very_High'])
        print("  - Created int_rate_category")
    
    # 7. Credit History Length
if 'earliest_cr_line' in df_features.columns:
        try:
            df_features['earliest_cr_line'] = pd.to_datetime(df_features['earliest_cr_line'])
            df_features['credit_history_length'] = (pd.Timestamp.now() - df_features['earliest_cr_line']).dt.days / 365.25
            print("  - Created credit_history_length")
        except:
            print("  - Could not create credit_history_length")
    
    # 8. Monthly Payment Ratio
if 'annual_inc' in df_features.columns and 'loan_amnt' in df_features.columns and 'int_rate' in df_features.columns:
        # Approximate monthly payment calculation
        monthly_income = df_features['annual_inc'] / 12
        monthly_rate = df_features['int_rate'] / 100 / 12
        
        # Simple monthly payment approximation
        df_features['payment_to_income_ratio'] = (df_features['loan_amnt'] / 36) / (monthly_income + 1)
        print("  - Created payment_to_income_ratio")

Creating domain-specific features...
  - Created debt_to_income_ratio
  - Created credit_utilization
  - Created emp_length_years
  - Created fico_avg and fico_range_width
  - Created loan_amount_category
  - Created int_rate_category
  - Created credit_history_length
  - Created payment_to_income_ratio


In [5]:
df_features.head()

,Unnamed: 0,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,target,debt_to_income_ratio,credit_utilization,emp_length_years,fico_avg,fico_range_width,loan_amount_category,int_rate_category,credit_history_length,payment_to_income_ratio
0,0,5000.0,5000.0,4975.0,36 months,10.65,162.87,B,B2,10+ years,...,0,0.208325,0.837,10.0,737.0,4.0,Small,Medium,40.520192,0.069410
1,1,2500.0,2500.0,2500.0,60 months,15.27,59.83,C,C4,< 1 year,...,1,0.083331,0.094,1.0,742.0,4.0,Small,High,26.275154,0.027767
2,2,2400.0,2400.0,2400.0,36 months,15.96,84.33,C,C5,10+ years,...,0,0.195870,0.985,10.0,737.0,4.0,Small,High,23.687885,0.065232
3,3,10000.0,10000.0,10000.0,36 months,13.49,339.31,C,C1,10+ years,...,0,0.203248,0.210,10.0,692.0,4.0,Small,Medium,29.437372,0.067734
4,4,3000.0,3000.0,3000.0,60 months,12.69,67.79,B,B5,1 year,...,0,0.037500,0.539,1.0,697.0,4.0,Small,Medium,29.522245,0.012498


In [3]:
X = df.drop(columns=["target"])
y = df["target"]

In [4]:
df.columns

Index(['Unnamed: 0', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term',
       'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
       'loan_status', 'pymnt_plan', 'purpose', 'addr_state', 'dti',
       'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv',
       'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
       'last_fico_range_high', 'last_fico_range_low',
       'collections_12_mths_ex_med', 'policy_code', 'application_type',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util'

In [5]:
categorical_cols = [
    'term',                 # e.g., '36 months', '60 months'
    'grade',                # A–G
    'sub_grade',            # A1–G5
    'emp_length',           # '< 1 year', '10+ years'
    'home_ownership',       # RENT, MORTGAGE, OWN, etc.
    'verification_status',  # Not Verified, Verified
    'purpose',              # debt_consolidation, credit_card, etc.
    'addr_state',           # state codes
    'initial_list_status',  # 'w' or 'f'
    'application_type',     # Individual, Joint App
    'hardship_flag',        # Y/N
    'debt_settlement_flag', # Y/N
    'pymnt_plan'            # usually 'n' or 'y'
]


In [13]:
df = df_features

In [14]:
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["issue_year"] = df["issue_d"].dt.year
df["issue_month"] = df["issue_d"].dt.month

In [7]:
# df.drop(columns=["issue_d"], inplace=True)


In [15]:
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")
df["credit_age_months"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days // 30
df["credit_age_months"] = df["credit_age_months"].clip(lower=0)  # Ensure non-negative


In [16]:
df.drop(columns=["last_pymnt_d", "last_credit_pull_d","issue_d","earliest_cr_line"], inplace=True)
df

,Unnamed: 0,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,emp_length_years,fico_avg,fico_range_width,loan_amount_category,int_rate_category,credit_history_length,payment_to_income_ratio,issue_year,issue_month,credit_age_months
0,0,5000.0,5000.0,4975.0,36 months,10.65,162.87,B,B2,10+ years,...,10.0,737.0,4.0,Small,Medium,40.520192,0.069410,2011,12,327
1,1,2500.0,2500.0,2500.0,60 months,15.27,59.83,C,C4,< 1 year,...,1.0,742.0,4.0,Small,High,26.275154,0.027767,2011,12,154
2,2,2400.0,2400.0,2400.0,36 months,15.96,84.33,C,C5,10+ years,...,10.0,737.0,4.0,Small,High,23.687885,0.065232,2011,12,122
3,3,10000.0,10000.0,10000.0,36 months,13.49,339.31,C,C1,10+ years,...,10.0,692.0,4.0,Small,Medium,29.437372,0.067734,2011,12,192
4,4,3000.0,3000.0,3000.0,60 months,12.69,67.79,B,B5,1 year,...,1.0,697.0,4.0,Small,Medium,29.522245,0.012498,2011,12,193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1860326,105446,24000.0,24000.0,24000.0,60 months,23.99,690.30,E,E2,< 1 year,...,1.0,672.0,4.0,Large,Very_High,30.275154,0.074758,2017,4,267
1860327,105447,10000.0,10000.0,10000.0,36 months,7.99,313.32,A,A5,10+ years,...,10.0,727.0,4.0,Small,Low,31.854894,0.051273,2017,4,287
1860328,105448,10050.0,10050.0,10050.0,36 months,16.99,358.26,D,D1,8 years,...,8.0,707.0,4.0,Medium,High,32.191650,0.090511,2017,4,291
1860329,105449,6000.0,6000.0,6000.0,36 months,11.44,197.69,B,B4,5 years,...,5.0,672.0,4.0,Small,Medium,35.192334,0.048766,2017,4,327


In [17]:
leakage_columns = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt', 'last_fico_range_high',
    'last_fico_range_low', 'loan_status', 'Unnamed: 0'
]

df = df.drop(columns=leakage_columns)


In [20]:
df['target']

0          0
1          1
2          0
3          0
4          0
          ..
1860326    1
1860327    0
1860328    1
1860329    0
1860330    1
Name: target, Length: 1860331, dtype: int64

In [21]:
# Define known columns to exclude
excluded_cols = [
    'Unnamed: 0', 'loan_status', 'target',               # Not useful or already encoded
    'term', 'grade', 'sub_grade', 'emp_length',          # Categorical
    'home_ownership', 'verification_status', 'purpose',
    'addr_state', 'initial_list_status', 'application_type',
    'hardship_flag', 'debt_settlement_flag', 'pymnt_plan'
]

categorical_cols = [
    'term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
    'verification_status', 'pymnt_plan', 'purpose', 'addr_state',
    'initial_list_status', 'application_type', 'hardship_flag',
    'debt_settlement_flag', 'loan_amount_category', 'int_rate_category',
    'issue_year', 'issue_month'
]


# Numerical columns = everything not in excluded
numerical_cols = [col for col in df.columns
                  if col not in excluded_cols and col not in categorical_cols]


In [22]:
print("Numerical columns (sample):", numerical_cols[:10])
print("Total numerical columns:", len(numerical_cols))


Numerical columns (sample): ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high']
Total numerical columns: 65


In [23]:
df.to_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/clean2.csv")

In [24]:
df_imputed = df.copy()
    
    # Separate numeric and categorical columns
numeric_cols = df_imputed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_imputed.select_dtypes(include=['object']).columns.tolist()
    
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
    

        # For numeric columns - use median for skewed, mean for normal
for col in numeric_cols:
            if df_imputed[col].isnull().sum() > 0:
                # Check skewness
                skewness = df_imputed[col].skew()
                if abs(skewness) > 1:  # Highly skewed
                    df_imputed[col].fillna(df_imputed[col].median(), inplace=True)
                else:  # Normal distribution
                    df_imputed[col].fillna(df_imputed[col].mean(), inplace=True)
        
        # For categorical columns - use mode
for col in categorical_cols:
            if df_imputed[col].isnull().sum() > 0:
                df_imputed[col].fillna(df_imputed[col].mode()[0], inplace=True)
    

    
print(f"Missing values after imputation: {df_imputed.isnull().sum().sum()}")

Numeric columns: 68
Categorical columns: 13
Missing values after imputation: 0


In [39]:
df_imputed.to_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/clean_credit_impute.csv")

In [27]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Only scale numeric and encode categoricals (no imputation)
num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])


In [28]:
X = df_imputed.drop(columns=["target"])
y = df_imputed["target"]


In [29]:
X

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,emp_length_years,fico_avg,fico_range_width,loan_amount_category,int_rate_category,credit_history_length,payment_to_income_ratio,issue_year,issue_month,credit_age_months
0,5000.0,5000.0,4975.0,36 months,10.65,162.87,B,B2,10+ years,RENT,...,10.0,737.0,4.0,Small,Medium,40.520192,0.069410,2011,12,327
1,2500.0,2500.0,2500.0,60 months,15.27,59.83,C,C4,< 1 year,RENT,...,1.0,742.0,4.0,Small,High,26.275154,0.027767,2011,12,154
2,2400.0,2400.0,2400.0,36 months,15.96,84.33,C,C5,10+ years,RENT,...,10.0,737.0,4.0,Small,High,23.687885,0.065232,2011,12,122
3,10000.0,10000.0,10000.0,36 months,13.49,339.31,C,C1,10+ years,RENT,...,10.0,692.0,4.0,Small,Medium,29.437372,0.067734,2011,12,192
4,3000.0,3000.0,3000.0,60 months,12.69,67.79,B,B5,1 year,RENT,...,1.0,697.0,4.0,Small,Medium,29.522245,0.012498,2011,12,193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1860326,24000.0,24000.0,24000.0,60 months,23.99,690.30,E,E2,< 1 year,RENT,...,1.0,672.0,4.0,Large,Very_High,30.275154,0.074758,2017,4,267
1860327,10000.0,10000.0,10000.0,36 months,7.99,313.32,A,A5,10+ years,MORTGAGE,...,10.0,727.0,4.0,Small,Low,31.854894,0.051273,2017,4,287
1860328,10050.0,10050.0,10050.0,36 months,16.99,358.26,D,D1,8 years,RENT,...,8.0,707.0,4.0,Medium,High,32.191650,0.090511,2017,4,291
1860329,6000.0,6000.0,6000.0,36 months,11.44,197.69,B,B4,5 years,RENT,...,5.0,672.0,4.0,Small,Medium,35.192334,0.048766,2017,4,327


In [30]:
X = preprocessor.fit_transform(X)

In [31]:
X

array([[-1.06887082, -1.06850681, -1.06916363, ...,  0.        ,
         1.        ,  0.        ],
       [-1.34756305, -1.34728134, -1.34510439, ...,  0.        ,
         1.        ,  0.        ],
       [-1.35871074, -1.35843232, -1.35625351, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.50591252, -0.50538224, -0.50334571, ...,  0.        ,
         1.        ,  0.        ],
       [-0.95739393, -0.95699699, -0.95488513, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.71805147,  1.71923856,  1.72090404, ...,  0.        ,
         1.        ,  0.        ]])

In [32]:
y

0          0
1          1
2          0
3          0
4          0
          ..
1860326    1
1860327    0
1860328    1
1860329    0
1860330    1
Name: target, Length: 1860331, dtype: int64

In [16]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import joblib

In [17]:
X = df.drop(columns=["target"])
y = df["target"]

In [19]:
# Remove % sign and convert to float
X["int_rate"] = X["int_rate"].str.strip().str.replace("%", "").astype(float)

In [20]:
# Detect columns with percentage values (object dtype + contains '%')
percent_cols = [col for col in X.columns if X[col].dtype == 'object' and X[col].astype(str).str.contains('%').any()]

# Clean and convert them to float
for col in percent_cols:
    X[col] = X[col].astype(str).str.strip().str.replace('%', '', regex=False).astype(float)

print(f"Cleaned percentage columns: {percent_cols}")

Cleaned percentage columns: ['revol_util']


In [21]:
def clean_percent_column(series):
    return series.apply(lambda x: float(str(x).replace('%', '').strip()) if isinstance(x, str) and '%' in x else x)


X['revol_util'] = clean_percent_column(X['revol_util'])


In [22]:
print(X.dtypes[X.dtypes == 'object'])


term                    object
grade                   object
sub_grade               object
emp_length              object
home_ownership          object
verification_status     object
pymnt_plan              object
purpose                 object
addr_state              object
initial_list_status     object
application_type        object
hardship_flag           object
debt_settlement_flag    object
dtype: object


In [23]:
# Numerical pipeline: Impute with median, then scale
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline: Impute with mode, then one-hot encode
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into a full preprocessor
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])


In [24]:
X_processed = preprocessor.fit_transform(X)
print("Processed shape:", X_processed.shape)

Processed shape: (1860331, 198)


In [43]:
X_processed

array([[-1.06887082, -1.06850681, -1.06916363, ...,  1.        ,
         0.        ,  1.        ],
       [-1.34756305, -1.34728134, -1.34510439, ...,  1.        ,
         0.        ,  1.        ],
       [-1.35871074, -1.35843232, -1.35625351, ...,  1.        ,
         0.        ,  1.        ],
       ...,
       [-0.50591252, -0.50538224, -0.50334571, ...,  1.        ,
         0.        ,  1.        ],
       [-0.95739393, -0.95699699, -0.95488513, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.71805147,  1.71923856,  1.72090404, ...,  1.        ,
         0.        ,  1.        ]])

In [39]:
y

0          0
1          1
2          0
3          0
4          0
          ..
1860326    1
1860327    0
1860328    1
1860329    0
1860330    1
Name: target, Length: 1860331, dtype: int64

In [33]:
import joblib

joblib.dump(X, "C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/X_processed_1.pkl")
joblib.dump(y, "C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y_1.pkl")


['C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y_1.pkl']

In [34]:
X_processed = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/X_processed_1.pkl")

y = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y.pkl")

In [35]:
X_processed

array([[-1.06887082, -1.06850681, -1.06916363, ...,  0.        ,
         1.        ,  0.        ],
       [-1.34756305, -1.34728134, -1.34510439, ...,  0.        ,
         1.        ,  0.        ],
       [-1.35871074, -1.35843232, -1.35625351, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.50591252, -0.50538224, -0.50334571, ...,  0.        ,
         1.        ,  0.        ],
       [-0.95739393, -0.95699699, -0.95488513, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.71805147,  1.71923856,  1.72090404, ...,  0.        ,
         1.        ,  0.        ]])